# Submit

Collect the fegrow mcs results + the default run and make a single submission filling the gaps in the mcs run using the default.

In [1]:
from rdkit import Chem
import pathlib
import pandas as pd
import ast

In [2]:
# =========================================================
# Create a SMILES : Label text file
# =========================================================
#
# INPUT FILES:
# 1. labels.txt
#    Contains:
#    ['MERS-CoV Mpro', 'SARS-CoV-2 Mpro', ...]
#
# 2. mers_smiles.txt
#    One MERS SMILES per line
#
# 3. sars2_smiles.txt
#    One SARS-CoV-2 SMILES per line
#
# OUTPUT:
# smiles_label.txt
#
# Format:
# SMILES<TAB>LABEL
#
# Example:
# CCO	MERS-CoV Mpro
# CCN	SARS-CoV-2 Mpro
#
# =========================================================



# ---------------------------------------------------------
# Step 1: Read labels file
# ---------------------------------------------------------

labels_file = "labels.txt"

with open(labels_file, "r", encoding="utf-8") as f:
    labels = ast.literal_eval(f.read())

print("Total labels loaded:", len(labels))

# ---------------------------------------------------------
# Step 2: Read MERS SMILES
# ---------------------------------------------------------

mers_file = "full_run-MERS/smiles-test-MERS.txt"

with open(mers_file, "r", encoding="utf-8") as f:
    mers_smiles = [line.strip() for line in f if line.strip()]

print("Total MERS SMILES:", len(mers_smiles))

# ---------------------------------------------------------
# Step 3: Read SARS-CoV-2 SMILES
# ---------------------------------------------------------

sars2_file = "sars_run_mcs/smiles-test-SARS.txt"

with open(sars2_file, "r", encoding="utf-8") as f:
    sars2_smiles = [line.strip() for line in f if line.strip()]

print("Total SARS-CoV-2 SMILES:", len(sars2_smiles))

# ---------------------------------------------------------
# Step 4: Create SMILES -> Label mapping
# ---------------------------------------------------------

mers_index = 0
sars2_index = 0

output_lines = []

for label in labels:

    # ---------------------------------------------
    # MERS-CoV
    # ---------------------------------------------
    if label == "MERS-CoV Mpro":

        if mers_index >= len(mers_smiles):
            raise ValueError(
                "Not enough SMILES in mers_smiles.txt"
            )

        smiles = mers_smiles[mers_index]

        output_lines.append(f"{smiles}\t{label}")

        mers_index += 1

    # ---------------------------------------------
    # SARS-CoV-2
    # ---------------------------------------------
    elif label == "SARS-CoV-2 Mpro":

        if sars2_index >= len(sars2_smiles):
            raise ValueError(
                "Not enough SMILES in sars2_smiles.txt"
            )

        smiles = sars2_smiles[sars2_index]

        output_lines.append(f"{smiles}\t{label}")

        sars2_index += 1

    # ---------------------------------------------
    # Unknown label
    # ---------------------------------------------
    else:
        print("Unknown label found:", label)

# ---------------------------------------------------------
# Step 5: Save to text file
# ---------------------------------------------------------

output_file = "smiles_label.txt"

with open(output_file, "w", encoding="utf-8") as f:

    # Optional header
    f.write("SMILES\tLABEL\n")

    for line in output_lines:
        f.write(line + "\n")

# ---------------------------------------------------------
# Step 6: Done
# ---------------------------------------------------------

print("\nText file created successfully!")
print("Saved as:", output_file)
print("Total entries written:", len(output_lines))

# ---------------------------------------------------------
# Step 7: Preview first 10 entries
# ---------------------------------------------------------

print("\nPreview:\n")

for line in output_lines[:10]:
    print(line)

Total labels loaded: 195
Total MERS SMILES: 97
Total SARS-CoV-2 SMILES: 98

Text file created successfully!
Saved as: smiles_label.txt
Total entries written: 195

Preview:

O=C1c2ccc(Cl)cc2[C@@]2(CN1Cc1nnco1)C(=O)N(c1cncc3ccccc13)C[C@@H]2CNc1ncncn1	MERS-CoV Mpro
COc1ccc(N(Cc2cccc(Cl)c2)C(=O)Cc2cncc3ccccc23)cc1	MERS-CoV Mpro
CNC(=O)CN1C[C@@]2(C(=O)N(c3cncc4ccccc34)C[C@@H]2CNc2ccncn2)c2cc(Cl)ccc2C1=O	SARS-CoV-2 Mpro
CNC(=O)CN1C[C@@]2(C(=O)N(c3cncc4ccccc34)C[C@@H]2[NH3+])c2cc(Cl)ccc2C1=O	MERS-CoV Mpro
O=C1c2ccc(Cl)cc2[C@@]2(CN1Cc1ccnnc1)C(=O)N(c1cncc3ccccc13)C[C@@H]2CNc1ccncn1	MERS-CoV Mpro
CNC(=O)CN1C[C@@]2(C(=O)N(c3cncc4ccccc34)C[C@@H]2CNc2cnn(C)c2)c2cc(Cl)ccc2C1=O	SARS-CoV-2 Mpro
Cc1cnc(CN2C[C@@]3(C(=O)N(c4cncc5ccccc45)C[C@@H]3C)c3cc(F)ccc3C2=O)cn1	SARS-CoV-2 Mpro
CNC(=O)CN1C[C@@]2(C(=O)N(c3cncc4ccccc34)C[C@@H]2COC(C)C)c2cc(Cl)ccc2C1=O	SARS-CoV-2 Mpro
C[C@H]1CN(c2cncc3ccccc23)C(=O)[C@@]12CN(Cc1nccn1C)C(=O)c1ccc(F)cc12	SARS-CoV-2 Mpro
C[C@H]1c2ccsc2CCN1C(=O)Cc1cncc2ccccc12	MERS-CoV Mpro

In [3]:
# ============================================================
# Create final_submission.sdf based on labels + best pockets
# ============================================================
#
# Logic:
#
# 1. Read smiles_label.txt
#
# 2. For each SMILES:
#
#    If label == "SARS-CoV-2 Mpro":
#         - check if molecule exists in:
#              sars_run_mcs/best_sars.sdf
#         - if YES -> use that molecule
#         - if NO  -> use molecule from:
#              default_fegrow/submitted_default.sdf
#
#    If label == "MERS-CoV Mpro":
#         - check if molecule exists in:
#              full_run-MERS/best_mers.sdf
#         - if YES -> use that molecule
#         - if NO  -> use molecule from:
#              default_fegrow/submitted_default.sdf
#
# 3. Save all selected molecules into:
#       final_submission.sdf
#
# ============================================================

from rdkit import Chem
from rdkit.Chem import AllChem

# ------------------------------------------------------------
# FILE PATHS
# ------------------------------------------------------------

LABEL_FILE = "smiles_label.txt"

SARS_SDF = "sars_run_mcs/best_sars.sdf"
MERS_SDF = "full_run-MERS/best_mers.sdf"

DEFAULT_SDF = "default_fegrow/submitted_default.sdf"

OUTPUT_SDF = "final_submission.sdf"

# ------------------------------------------------------------
# FUNCTION:
# Load molecules from SDF into dictionary by canonical SMILES
# ------------------------------------------------------------

def load_sdf_as_dict(sdf_file):

    mol_dict = {}

    supplier = Chem.SDMolSupplier(sdf_file)

    for mol in supplier:

        if mol is None:
            continue

        try:
            # ignore stereochemistry
            smiles = Chem.MolToSmiles(
                mol,
                canonical=True,
                isomericSmiles=False
            )

            if smiles not in mol_dict:
                mol_dict[smiles] = mol

        except Exception:
            continue

    return mol_dict

# ------------------------------------------------------------
# LOAD ALL SDF FILES
# ------------------------------------------------------------

print("Loading SDF files...")

sars_dict = load_sdf_as_dict(SARS_SDF)
mers_dict = load_sdf_as_dict(MERS_SDF)
default_dict = load_sdf_as_dict(DEFAULT_SDF)

print("SARS molecules loaded:", len(sars_dict))
print("MERS molecules loaded:", len(mers_dict))
print("Default molecules loaded:", len(default_dict))

# ------------------------------------------------------------
# READ SMILES + LABELS
# ------------------------------------------------------------

smiles_labels = []

with open(LABEL_FILE, "r", encoding="utf-8") as f:

    next(f)  # skip header

    for line in f:

        line = line.strip()

        if not line:
            continue

        smiles, label = line.split("\t")

        smiles_labels.append((smiles, label))

print("Labels loaded:", len(smiles_labels))

# ------------------------------------------------------------
# CREATE FINAL SUBMISSION
# ------------------------------------------------------------

writer = Chem.SDWriter(OUTPUT_SDF)

used_from_sars = 0
used_from_mers = 0
used_from_default = 0
missing_molecules = 0

for smiles, label in smiles_labels:

    # canonicalize input smiles
    mol_from_smiles = Chem.MolFromSmiles(smiles)

    if mol_from_smiles is None:
        print(f"Invalid SMILES: {smiles}")
        missing_molecules += 1
        continue

    canonical_smiles = Chem.MolToSmiles(
        mol_from_smiles,
        canonical=True,
        isomericSmiles=False
)

    selected_mol = None

    # --------------------------------------------------------
    # SARS CASE
    # --------------------------------------------------------

    if label == "SARS-CoV-2 Mpro":

        # Use best_sars.sdf if available
        if canonical_smiles in sars_dict:

            selected_mol = sars_dict[canonical_smiles]
            used_from_sars += 1

        # otherwise use default
        elif canonical_smiles in default_dict:

            selected_mol = default_dict[canonical_smiles]
            used_from_default += 1

    # --------------------------------------------------------
    # MERS CASE
    # --------------------------------------------------------

    elif label == "MERS-CoV Mpro":

        # Use best_mers.sdf if available
        if canonical_smiles in mers_dict:

            selected_mol = mers_dict[canonical_smiles]
            used_from_mers += 1

        # otherwise use default
        elif canonical_smiles in default_dict:

            selected_mol = default_dict[canonical_smiles]
            used_from_default += 1

    # --------------------------------------------------------
    # WRITE MOLECULE
    # --------------------------------------------------------

    if selected_mol is not None:

        # add metadata
        selected_mol.SetProp("SMILES", canonical_smiles)
        selected_mol.SetProp("LABEL", label)

        writer.write(selected_mol)

    else:

        print(f"Molecule not found anywhere: {canonical_smiles}")
        missing_molecules += 1

# ------------------------------------------------------------
# CLOSE WRITER
# ------------------------------------------------------------

writer.close()

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n====================================")
print("FINAL SUBMISSION CREATED")
print("====================================")

print("Used from SARS best sdf :", used_from_sars)
print("Used from MERS best sdf :", used_from_mers)
print("Used from default sdf   :", used_from_default)
print("Missing molecules       :", missing_molecules)

print(f"\nSaved to: {OUTPUT_SDF}")

Loading SDF files...
SARS molecules loaded: 83
MERS molecules loaded: 85
Default molecules loaded: 159
Labels loaded: 195

FINAL SUBMISSION CREATED
Used from SARS best sdf : 94
Used from MERS best sdf : 87
Used from default sdf   : 14
Missing molecules       : 0

Saved to: final_submission.sdf


In [7]:
# check that the smiles of stereoisomers are read as the same mol 
#(smiles_label.txt contains a smiles for a stereoisomer to sars molecule 151 has a slightly different smiles when loaded from sdf)

#smiles_x = "O=C1N[C@]2(C[C@H](c3ccccc3Cl)C2)C(=O)N1c1cncc2ccccc12"  # smiles in smiles_labels.txt
smiles_x = "O=C1N[C@]2(C[C@@H](c3ccccc3Cl)C2)C(=O)N1c1cncc2ccccc12"  # smiles from best_sars.sdf
mol = Chem.MolFromSmiles(smiles_x)

key = Chem.MolToSmiles(
    mol,
    canonical=True,
    isomericSmiles=False
)

if key in sars_dict:
    print("Found in sars_dict")
else:
    print("Not found in sars_dict")

Found in sars_dict
